# ALMs — Perplexity Attribution & Token-Level Analysis (Ch.5)

This notebook is **stage 2 of the ALMs method** (Ch.5 Sec 5.3.2–5.3.3): given
the per-author GPT-2s trained by `ALMs_Train.ipynb`, decide *who wrote a
questioned document* and explain *which words drove the decision*.

**The attribution rule is argmin perplexity.** For each candidate author
we ask their ALM: "how predictable is this text to you?" The perplexity
(`PPL = exp(mean per-token NLL)`) summarises the answer; the author whose
model gives the *lowest* PPL wins.

**Three scoring stages** (each a function in `thesis_aa.alms.ppl`):

1. `score_all_pairs` — score every (ALM, test-author) pair, writing
   per-token cross-entropy logs + a PPL table;
2. `aggregate_ppl` — turn the per-token logs into one PPL per test text
   (for several text-length cutoffs);
3. `predict_and_benchmark` — attribute each test text by argmin-PPL and
   compute the benchmark metrics.

Then a fourth function, `compute_cnll`, implements the token-level
**Comparative NLL** used for interpretability (Ch.5 Eq. 3–4).

> **Provenance note.** This module is ported from the reference repo's
> `CalculatePPL.ipynb` with **7 bugs fixed** (the original did not run
> end-to-end as published) — see the `thesis_aa/alms/ppl.py` module
> docstring for the audit trail. Bug #6 (double-counted context tokens in
> overlapping sliding windows) and #7 (grouping by `text_num` alone, which
> conflated different authors' documents) are the ones that corrupted the
> original's accuracy numbers.

## 0. Bootstrap + ensure trained models exist

We need the per-author ALMs from `ALMs_Train.ipynb`. To keep this notebook
self-contained, the cell below trains them *if* `models/` is empty — at
the demo's 15 epochs that costs ~10 minutes per author. (If you already
ran the training notebook, the skip-check fires and nothing retrains.)

In [1]:
import os, sys

REPO_ROOT = os.path.dirname(os.path.abspath(os.getcwd()))  # notebooks/ -> repo root
if REPO_ROOT not in sys.path:
    sys.path.insert(0, REPO_ROOT)

from thesis_aa import config, data as data_mod, eval as eval_mod
from thesis_aa.alms import train as alms_train, ppl as alms_ppl

# Three-author subset of the shipped natural demo corpus (same subset as
# ALMs_Train.ipynb, so models/ and this test set match).
full_train, full_test = data_mod.load_natural()
keep = {'author00', 'author01', 'author02'}
train_df = full_train[full_train['author_tag'].isin(keep)].reset_index(drop=True)
test_df = full_test[full_test['author_tag'].isin(keep)].reset_index(drop=True)

trained = [d for d in os.listdir(config.MODEL_DIR)
           if os.path.isdir(os.path.join(config.MODEL_DIR, d))]
if not trained:
    print('models/ is empty - training demo ALMs first (~10 min per author)...')
    alms_train.train_all_authors(
        train_df, epochs=15, gradient_accumulation_steps=1,
        batch_size=1, block_size=64, fp16=False)
else:
    print('Found existing models:', sorted(trained))

print('device:', config.get_device(), '| train:', train_df.shape, '| test:', test_df.shape)


C:\Users\MiraMoe\AppData\Local\Programs\Python\Python313\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Skipping import of cpp extensions due to incompatible torch version. Please upgrade to torch >= 2.11.0 (found 2.9.0+xpu).


W0904 13:52:39.565000 21628 site-packages\torch\distributed\elastic\multiprocessing\redirects.py:29] NOTE: Redirects are currently not supported in Windows or MacOs.


Found existing models: ['author00', 'author01', 'author02']
device: xpu | train: (150, 2) | test: (60, 2)


**What you should see:** either a quick training run (three authors,
2 epochs each) or the list of already-trained models, then the natural
demo corpus shapes. Note `use_tagger=False` by default in scoring; we
enable spaCy later for the annotation demo.


## 1. Score every (model, author) pair

`score_all_pairs` builds the **n_models × n_authors scoring matrix**: cell
(i, j) answers *"how predictable is author j's test text under author i's
model?"* Each cell is a mean perplexity over that author's test documents.

Two artifacts are written per pair:

- `results/ce_log/<model_tag>-<text_tag>.csv.7z` — an LZMA-compressed CSV
  holding *per-token* data: the token strings, each token's NLL under that
  model, and (optionally) spaCy annotations. One row per test text.
- `results/ppl_result.csv` — one summary row per pair
  (`model_tag, text_tag, stride, ppl`). This file doubles as the
  **resumability log**: pairs already present are skipped on restart.

In [2]:
result_path = alms_ppl.score_all_pairs(
    train_df, test_df,
    model_dir=config.MODEL_DIR,
    use_tagger=False,          # spaCy annotations off for now (demo in step 3)
    limit_texts_per_author=None,
)
print('PPL table written to:', result_path)

[ALMs/PPL] skip author00 x author00 (logged)
[ALMs/PPL] skip author00 x author01 (logged)
[ALMs/PPL] skip author00 x author02 (logged)


[ALMs/PPL] skip author01 x author00 (logged)
[ALMs/PPL] skip author01 x author01 (logged)
[ALMs/PPL] skip author01 x author02 (logged)


[ALMs/PPL] skip author02 x author00 (logged)
[ALMs/PPL] skip author02 x author01 (logged)
[ALMs/PPL] skip author02 x author02 (logged)
PPL table written to: D:\AgentHome\Thesis\results\ppl_result.csv


**What you should see:** progress like
`[ALMs/PPL] author00 x author00: PPL=X.XX` — 3 models × 3 test authors =
9 pairs. The diagonal (own model on own texts) should tend to score lower
than off-diagonal cells — that is the entire premise of the method.

In [3]:
import pandas as pd

ppl_df = pd.read_csv(result_path)
ppl_df

,model_tag,text_tag,stride,ppl
0,author00,author00,128,272.596920
1,author00,author01,128,298.484085
2,author00,author02,128,333.576837
3,author01,author00,128,451.146586
4,author01,author01,128,207.425519
5,author01,author02,128,324.942755
6,author02,author00,128,422.637516
7,author02,author01,128,285.727862
8,author02,author02,128,225.188894


### The scoring matrix, visualised

Pivot the table so rows = ALMs, columns = true test authors. The argmin
attribution rule says: for the texts in *column j*, the predicted author is
the *row* with the lowest value.

In [4]:
matrix = ppl_df.pivot(index='model_tag', columns='text_tag', values='ppl')
print('Perplexity matrix (rows = ALM, columns = true author):')
display(matrix.round(2))
print()
print('Argmin attribution (predicted author per true author):')
print(matrix.idxmin())

Perplexity matrix (rows = ALM, columns = true author):


text_tag,author00,author01,author02
model_tag,,,
author00,272.60,298.48,333.58
author01,451.15,207.43,324.94
author02,422.64,285.73,225.19


Argmin attribution (predicted author per true author):
text_tag
author00    author00
author01    author01
author02    author02
dtype: object


**What you should see:** a 3×3 perplexity matrix. Read it *column-wise*:
each column is one text author, and the method attributes their texts to
the row model with the lowest PPL (argmin down the column). After 15 further-pretraining epochs the
models have moved apart from the shared base model, and the diagonal
dominates: each author's texts are most predictable (lowest PPL) under
that author's own model. The
diaEach author's texts therefore score lowest under that author's own
model, and argmin attribution follows the diagonal (macro-accuracy
≈ 0.90 at 3 candidates in the demo).

That is the *directional* thesis behaviour already at demo scale: the
thesis's attribution headline (88.1% mean macro-accuracy, Ch.5 Table
5.2) comes from 100 further-pretraining epochs on corpora with far
more text per author and 50 candidate authors (Ch.5 Table 5.1). Raise
`epochs` toward the thesis configuration and re-run these two
notebooks to watch the diagonal sharpen further.


## 2. Open a per-token CE log

Each `.csv.7z` archive holds the raw per-token evidence behind the matrix
cells. Reading one shows the `FEATURE_CATEGORIES` schema: the token
strings, their per-token NLLs, and (empty) annotation columns when no
tagger was used. The lists are stored *stringified* (`"['the', ...]"`)
so the CSV stays a plain CSV; `ast.literal_eval` recovers them.

In [5]:
import ast, zipfile

pair = 'author00-author00'  # author00's model, scoring author00's texts
with zipfile.ZipFile(os.path.join(config.RESULTS_DIR, 'ce_log', pair + '.csv.7z')) as z:
    inner = [n for n in z.namelist() if n.endswith('.csv')][0]
    raw = pd.read_csv(z.open(inner), names=alms_ppl.FEATURE_CATEGORIES)

print('rows (test texts):', len(raw))
row0 = raw.iloc[0]
tokens = ast.literal_eval(row0['tokens'])
losses = ast.literal_eval(row0['losses'])
per_tok = pd.DataFrame({'token': tokens[1:], 'NLL': losses})  # losses align to tokens[1:]
display(per_tok.head(12))

rows (test texts): 20


,token,NLL
0,ights,5.659475
1,gathered,7.896259
2,in,1.514563
3,the,0.361180
4,great,0.721704
5,hall,0.137354
6,",",0.604245
7,and,1.417880
8,their,3.076128
9,banners,0.217789


**What you should see:** a token/NLL table for the first test text.
High NLL = the model was surprised by that token. Note the list lengths:
`len(tokens) - 1 == len(losses)` — the *first* token has no preceding
context, so there is nothing to predict it; losses are aligned to
`tokens[1:]`. These per-token losses are exactly what `aggregate_ppl`
re-reads and averages into text-level PPLs — and they're the substrate for
the CNLL analysis in step 5. Common words ("the", "of") get low NLL;
author-specific lexicon words get higher NLL under other authors' models.

## 3. Linguistic annotations (spaCy, optional)

`compute_ce_per_text` can tag every GPT-2 token on the fly with spaCy
(lemma, POS, shape, …) — used in the thesis to break down attribution
evidence by linguistic category. Pass `use_tagger=True` to
`score_all_pairs`, or call the per-text function directly as below. The
spaCy model (`python -m spacy download en_core_web_sm`) is optional; when
missing, annotation columns simply stay empty and the pipeline is
unaffected.

In [6]:
tagger = None
try:
    tagger = alms_ppl._spacy_pipeline()
    print('spaCy pipeline:', tagger.__class__.__name__, '| component names:', tagger.pipe_names)
except Exception as e:
    print('spaCy unavailable - annotations will stay empty:', e)

import torch
from transformers import AutoTokenizer, GPT2LMHeadModel

device = config.get_device()
model = GPT2LMHeadModel.from_pretrained(os.path.join(config.MODEL_DIR, 'author00')).to(device)
tokenizer = AutoTokenizer.from_pretrained(os.path.join(config.MODEL_DIR, 'author00'))

sample = test_df[test_df['author_tag'] == 'author01']['text'].iloc[0]
rec_tagged = alms_ppl.compute_ce_per_text(sample, model, tokenizer, device, tagger=tagger)

if tagger is not None:
    ann = pd.DataFrame({k: rec_tagged[k] for k in ['tokens', 'lemmas', 'poss', 'tags', 'stops']})
    display(ann.head(10))
else:
    print('No tagger available - showing tokens/losses only:')
    display(pd.DataFrame({'token': rec_tagged['tokens'][:10],
                          'NLL': rec_tagged['losses'][:10]}))
del model

spaCy pipeline: English | component names: ['tok2vec', 'tagger', 'parser', 'attribute_ruler', 'lemmatizer', 'ner']


`loss_type=None` was set in the config but it is unrecognized. Using the default loss: `ForCausalLMLoss`.


,tokens,lemmas,poss,tags,stops
0,The,the,DET,DT,True
1,research,research,NOUN,NN,False
2,vessel,vessel,NOUN,NN,False
3,anchored,anchore,VERB,VBD,False
4,over,over,INTJ,UH,False
5,the,the,DET,DT,False
6,deep,deep,PROPN,NNP,False
7,basin,basin,PROPN,NNP,False
8,at,at,NOUN,NNS,False
9,first,first,PROPN,NNP,False


**What you should see:** the same token stream with filled `lemmas`,
`poss`, `tags`, `stops` columns. The thesis uses these annotations to
aggregate per-token losses by POS class or stopword status and analyse
*what kind* of tokens carry authorship signal (Ch.5 Sec 5.4).

## 4. Aggregate per-text PPL and benchmark

\ggregate_ppl\ reads all CE logs back, computes one PPL per test text per
candidate, and writes a tidy dataframe (\ppl_dfs_buffer.csv\).

Then \predict_and_benchmark\ attributes each \(true_tag, text_num)\ group
by argmin-PPL (grouping by *both* keys — bug #7 in the original notebook
grouped by \	ext_num\ alone and silently conflated different authors'
documents) and writes accuracy per author.


In [7]:
ppl_paths = alms_ppl.aggregate_ppl()
agg = pd.read_csv(ppl_paths[0])
print('buffer:', ppl_paths[0])
display(agg.head(8))


[ALMs/PPL] wrote D:\AgentHome\Thesis\results\ppl_dfs_buffer\ppl_dfs_buffer.csv
buffer: D:\AgentHome\Thesis\results\ppl_dfs_buffer\ppl_dfs_buffer.csv


,true_tag,candidate_tag,text_num,global-ppl:(losses_shifted),global-ppl:(losses)
0,author00,author00,0,152.016801,152.016801
1,author00,author00,1,368.444133,368.444133
2,author00,author00,2,383.708902,383.708902
3,author00,author00,3,244.289777,244.289777
4,author00,author00,4,1514.000056,1514.000056
5,author00,author00,5,172.512422,172.512422
6,author00,author00,6,136.848028,136.848028
7,author00,author00,7,234.651155,234.651155


**What you should see:** the buffer with columns
`true_tag, candidate_tag, text_num, global-ppl:(losses_shifted),
global-ppl:(losses)` — for each test text, one row per candidate ALM. Two
"features" are stored: `losses` (the PPL as scored) and `losses_shifted`
(the PPL when each token's loss is attributed to the *next* position — a
one-position shift the original notebook carried for its alignment
convention; both are reported in the benchmark).

In [8]:
bench_paths = alms_ppl.predict_and_benchmark(ppl_paths)
print('benchmark CSVs:', [os.path.basename(p) for p in bench_paths])

bench = pd.read_csv(bench_paths[-1])   # full-length benchmark
print()
print('GLOBAL rows (overall metrics per feature):')
display(bench[bench['true_tag'] == 'GLOBAL'])

[ALMs/PPL] benchmark -> D:\AgentHome\Thesis\results\benchmark_results_df_home\benchmark_results_df_buffer.csv
benchmark CSVs: ['benchmark_results_df_buffer.csv']
GLOBAL rows (overall metrics per feature):


,feature,true_tag,accuracy
0,global-ppl:(losses),GLOBAL,0.9
4,global-ppl:(losses_shifted),GLOBAL,0.9


**What you should see:** for each feature (`global-ppl:(losses_shifted)`
and `global-ppl:(losses)`), a GLOBAL row plus one row per author with
accuracy. At the demo's 15-epoch training the accuracies reach
≈ 0.90 macro (chance would be ≈ 0.33) — attribution by lowest
perplexity works: predictions are made per text by argmin-PPL, then
scored globally and per author exactly as the thesis's benchmark tables
are built. With the thesis configuration (100 epochs, real corpora) the
same cells yield the reported 88.1% mean macro-accuracy.


In [9]:
# Thesis-style summary: macro-average over the per-author rows, per length.
summary = eval_mod.summarize_benchmark_dir(
    os.path.join(config.RESULTS_DIR, 'benchmark_results_df_home'))
display(summary)

,file,feature,macro_accuracy
0,benchmark_results_df_buffer.csv,global-ppl:(losses),0.9
1,benchmark_results_df_buffer.csv,global-ppl:(losses_shifted),0.9


The summary table averages each benchmark's per-author accuracies
into one **macro-accuracy** — the same aggregation the thesis reports per
dataset. On the real benchmarks this is where the 88.1% mean macro-accuracy
(Ch.5 Table 5.2) comes from.


## 5. Token-level interpretability — CNLL

The final piece of ALMs is explaining *why* a document was attributed to an
author. The **Comparative NLL** (Ch.5 Eq. 3–4) of a token for candidate *a*
is:

```
CNLL(a, i) = NLL_a(i) - mean_{b != a} NLL_b(i)
```

Negative ⇒ the token is *more predictable under a's model than under the
others* ⇒ it pushes attribution toward a. Positive ⇒ it pushes away. This
is the interpretability advantage over black-box classifiers: you can point
at the exact words that decided the case.

To build the `(n_tokens-1, n_authors)` NLL matrix we run every ALM over the
same questioned text — one forward pass per model — using
`compute_ce_per_text`:

In [10]:
import numpy as np

model_tags = sorted(d for d in os.listdir(config.MODEL_DIR)
                    if os.path.isdir(os.path.join(config.MODEL_DIR, d)))
questioned = test_df.iloc[0]['text']
print('Questioned text (true author:', test_df.iloc[0]['author_tag'], ')')
print(questioned[:120], '...\n')

nll_cols, toks = {}, None
for tag in model_tags:
    m = GPT2LMHeadModel.from_pretrained(os.path.join(config.MODEL_DIR, tag)).to(device)
    tok = AutoTokenizer.from_pretrained(os.path.join(config.MODEL_DIR, tag))
    rec = alms_ppl.compute_ce_per_text(questioned, m, tok, device)
    toks = rec['tokens']
    nll_cols[tag] = rec['losses']   # unpadded: length len(tokens)-1, aligned to tokens[1:]
    del m

nll_matrix = np.column_stack([nll_cols[t] for t in model_tags])
print('NLL matrix shape (n_tokens-1, n_authors):', nll_matrix.shape)

Questioned text (true author: author00 )
<BOS>Knights gathered in the great hall, and their banners hung from every rafter while torches guttered in the drafts.
 ...


NLL matrix shape (n_tokens-1, n_authors): (294, 3)


Now `compute_cnll` compares the candidate columns. With
`candidate=None` it automatically picks the **predicted author** (argmin of
total NLL) — matching how the thesis reports the evidence for whoever was
attributed. The alignment is one-to-one: row `i` of `nll_matrix` is the NLL
of `tokens[i+1]`, and `cnll[i]` belongs to the same token — so the evidence
table pairs `tokens[1:]` with `cnll`:

In [11]:
cnll = alms_ppl.compute_cnll(nll_matrix, model_tags)  # candidate=None -> predicted author
predicted = model_tags[int(np.argmin(nll_matrix.sum(axis=0)))]
print(f'Predicted author (lowest total NLL): {predicted}')
print(f'True author: {test_df.iloc[0]["author_tag"]}\n')

evidence = pd.DataFrame({'token': toks[1:], 'CNLL': cnll})
evidence = evidence.sort_values('CNLL').reset_index(drop=True)
print('Tokens most favouring', predicted, '(most negative CNLL):')
display(evidence.head(8))
print()
print('Tokens most against', predicted, '(most positive CNLL):')
display(evidence.tail(5))

Predicted author (lowest total NLL): author00
True author: author00
Tokens most favouring author00 (most negative CNLL):


,token,CNLL
0,drafts,-14.938874
1,torches,-10.665193
2,gut,-10.385749
3,banners,-9.688513
4,admittedly,-9.556866
5,while,-9.075036
6,admittedly,-8.581501
7,Someone,-8.538717


Tokens most against author00 (most positive CNLL):


,token,CNLL
289,thunder,4.491043
290,dale,4.574436
291,sudden,4.942373
292,move,5.706904
293,blasted,7.728049


**What you should see:** a ranked token table. Negative-CNLL tokens are
the document's authorial "fingerprints" under the winning model — tokens
that model expected better than the average of the other candidates. At
demo scale the margins are small, but the decomposition is exactly the
thesis's (Ch.5 Sec 5.3.3): attribution becomes *auditable* — you can show
a human precisely which tokens drove the decision, and on fully trained
ALMs (Ch.5 Sec 5.4.3) those tokens are interpretable as authorial habits.


## 6. Going to real data

The same three calls, on a real benchmark with 100-epoch models, reproduce
the thesis's ALMs numbers:

\\python
train_df, test_df = data_mod.load_benchmark('Blogs50')   # 50 candidate authors
result_path = alms_ppl.score_all_pairs(train_df, test_df, model_dir=config.MODEL_DIR)
ppl_paths = alms_ppl.aggregate_ppl()
bench_paths = alms_ppl.predict_and_benchmark(ppl_paths)
summary = eval_mod.summarize_benchmark_dir(os.path.join(config.RESULTS_DIR, 'benchmark_results_df_home'))
\
| Aspect | Demo | Thesis |
|---|---|---|
| Candidates | 3 author personas | 50 real authors |
| Epochs per ALM | 15 | 100 |
| Headline metric | ≈ 0.90 macro-accuracy (3 authors, demo) | 88.1% mean macro-accuracy |

For a full reproduction: train with \config.ALMS_TRAIN_CONFIG(\ALMs_Train.ipynb\, step 5) and score with \stride=128\ (default).
\score_all_pairs\ is resumable via esults/ppl_result.csv\, so a long
scoring run can be interrupted and restarted safely.
